In [0]:
dfr = spark.table("ifood.rejected.rejected_data")

In [0]:
dfr.groupBy("origem").count().show()

+------+-------+
|origem|  count|
+------+-------+
|yellow|4283308|
| green|  31617|
+------+-------+



In [0]:
dfg = spark.table("ifood.trusted.green_taxi")
dfy = spark.table("ifood.trusted.yellow_taxi")

In [0]:
dfg.show(n=2)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+-------------+----+-----+-----------+------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|      id_data|year|month|erro_concat|origem|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+-------------+----+

In [0]:
dfy.show(n=2)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------+----+-----+-----------+------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|       id_data|year|month|erro_concat|origem|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------+----+-----+-----------+---

In [0]:
dfg = dfg.select("VendorID", "passenger_count", "tpep_pickup_datetime", "tpep_dropoff_datetime", "total_amount", "id_data", "year", "month", "origem")
dfy = dfy.select("VendorID", "passenger_count", "tpep_pickup_datetime", "tpep_dropoff_datetime", "total_amount", "id_data", "year", "month", "origem")

df = dfg.unionByName(dfy)

In [0]:
df = (
    df
    .withColumnRenamed(
        "tpep_pickup_datetime",
        "pickup_datetime"
    )
    .withColumnRenamed(
        "tpep_dropoff_datetime",
        "dropoff_datetime"
    )
)

In [0]:
display(df)

VendorID,passenger_count,pickup_datetime,dropoff_datetime,total_amount,id_data,year,month,origem
2,1,2026-03-01T00:46:53.000,2026-03-01T00:48:34.000,52.0,green_2026_03,2026,03,green
2,1,2026-03-01T00:06:37.000,2026-03-01T00:38:51.000,79.56,green_2026_03,2026,03,green
2,1,2026-03-01T00:04:05.000,2026-03-01T00:30:26.000,47.6,green_2026_03,2026,03,green
2,1,2026-03-01T00:32:52.000,2026-03-01T00:50:06.000,20.2,green_2026_03,2026,03,green
2,1,2026-03-01T00:22:30.000,2026-03-01T00:27:33.000,11.64,green_2026_03,2026,03,green
2,1,2026-03-01T00:38:03.000,2026-03-01T00:49:18.000,20.88,green_2026_03,2026,03,green
1,1,2026-03-01T00:35:34.000,2026-03-01T00:35:59.000,80.0,green_2026_03,2026,03,green
2,1,2026-03-01T00:07:14.000,2026-03-01T00:16:30.000,13.9,green_2026_03,2026,03,green
2,1,2026-03-01T00:43:35.000,2026-03-01T01:03:40.000,37.7,green_2026_03,2026,03,green
2,1,2026-03-01T00:33:45.000,2026-03-01T00:52:51.000,30.08,green_2026_03,2026,03,green


In [0]:
path_consumption = "s3://ifood-case-data-quality/consumption-data/"
(df.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("year", "month")
    .save(path_consumption)
)